In [2]:
!pip install --user albumentations opencv-python pyyaml tqdm

  Using cached albumentations-2.0.8-py3-none-any.whl.metadata (43 kB)
  Using cached albucore-0.0.24-py3-none-any.whl.metadata (5.3 kB)
  Using cached opencv_python_headless-4.13.0.92-cp37-abi3-win_amd64.whl.metadata (20 kB)
Using cached albumentations-2.0.8-py3-none-any.whl (369 kB)
Using cached albucore-0.0.24-py3-none-any.whl (15 kB)
Using cached opencv_python_headless-4.13.0.92-cp37-abi3-win_amd64.whl (40.1 MB)

   ---------------------------------------- 0/3 [opencv-python-headless]
   ---------------------------------------- 0/3 [opencv-python-headless]
   ---------------------------------------- 0/3 [opencv-python-headless]
   ---------------------------------------- 0/3 [opencv-python-headless]
   ---------------------------------------- 0/3 [opencv-python-headless]
   ---------------------------------------- 0/3 [opencv-python-headless]
   -------------------------- ------------- 2/3 [albumentations]
   -------------------------- ------------- 2/3 [albumentations]
   ---------


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import os
import cv2
import numpy as np
import random
import shutil
from pathlib import Path
from tqdm import tqdm
import albumentations as A
from albumentations.pytorch import ToTensorV2
import yaml
import json


RAW_ROOT = Path(r"C:\Users\IvanC\kurs_work\processed\AOD4_clean")
AUG_ROOT = Path(r"C:\Users\IvanC\kurs_work\processed\AOD4_augmented")

AUGMENT_SPLITS = ["train"]

AUG_LEVEL = "light"

AUG_COPIES_PER_IMAGE = 1

random.seed(42)
np.random.seed(42)

BIRD_SOURCES_DIR = RAW_ROOT / "images" / "train"

def parse_yolo_label(lbl_path):
    objects = []
    if not lbl_path.exists():
        return objects
    with open(lbl_path, 'r', encoding='utf-8') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) != 5:
                continue
            cls, xc, yc, w, h = map(float, parts)
            objects.append({
                "class_id": int(cls),
                "xc": xc, "yc": yc, "w": w, "h": h
            })
    return objects

def write_yolo_label(lbl_path, objects):
    with open(lbl_path, 'w', encoding='utf-8') as f:
        for obj in objects:
            f.write(f"{obj['class_id']} {obj['xc']:.6f} {obj['yc']:.6f} {obj['w']:.6f} {obj['h']:.6f}\n")

def get_random_bird_patch():
    bird_imgs = list(BIRD_SOURCES_DIR.glob("*.jpg")) + list(BIRD_SOURCES_DIR.glob("*.png"))
    if not bird_imgs:
        return None, None, None

    for _ in range(10):
        src_img_path = random.choice(bird_imgs)
        src_lbl_path = (RAW_ROOT / "labels" / "train" / f"{src_img_path.stem}.txt")
        birds = parse_yolo_label(src_lbl_path)
        bird_objs = [b for b in birds if b["class_id"] == 1]
        
        if bird_objs:
            bird = random.choice(bird_objs)
            src_img = cv2.imread(str(src_img_path))
            if src_img is None:
                continue
                
            h, w = src_img.shape[:2]
            x1 = int((bird["xc"] - bird["w"]/2) * w)
            y1 = int((bird["yc"] - bird["h"]/2) * h)
            x2 = int((bird["xc"] + bird["w"]/2) * w)
            y2 = int((bird["yc"] + bird["h"]/2) * h)
            
            x1, y1 = max(0, x1), max(0, y1)
            x2, y2 = min(w, x2), min(h, y2)
            
            patch = src_img[y1:y2, x1:x2]
            if patch.size == 0:
                continue
            return patch, bird["h"] * h / src_img.shape[0], bird["w"] * w / src_img.shape[1]
    return None, None, None

class AugmentationPipeline:
    def __init__(self, level="medium"):
        self.level = level
        self.transforms = self._build_pipeline()

    def _build_pipeline(self):
        
        light = A.Compose([
            A.OneOf([
                A.RandomBrightnessContrast(brightness_limit=0.3, contrast_limit=0.3, p=0.6),
                A.HueSaturationValue(hue_shift_limit=20, sat_shift_limit=30, val_shift_limit=20, p=0.5),
            ], p=0.8),
            A.GaussNoise(var_limit=(10.0, 30.0), p=0.4),
            A.Blur(blur_limit=3, p=0.2),
            A.HorizontalFlip(p=0.5),
            A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1, rotate_limit=15, p=0.4),
        ], bbox_params=A.BboxParams(format='yolo', label_fields=['class_ids']))

        medium = A.Compose([
            *light.transforms,
            A.ImageCompression(quality_lower=40, quality_upper=70, p=0.4),
            A.MotionBlur(blur_limit=5, p=0.2),
            A.RandomFog(fog_coef_lower=0.1, fog_coef_upper=0.3, alpha_coef=0.1, p=0.25),
            A.RandomRain(drop_length=15, drop_width=1, drop_color=(180, 180, 180), p=0.2),
            A.RandomSnow(brightness_coeff=1.1, snow_point_lower=0.1, snow_point_upper=0.25, p=0.2),
        ], bbox_params=A.BboxParams(format='yolo', label_fields=['class_ids']))

        hard = A.Compose([
            *medium.transforms,
            A.GaussNoise(var_limit=(20.0, 50.0), p=0.5),
            A.Defocus(radius=(2, 5), alias_blur=(0.2, 0.4), p=0.3),
            A.Spatter(mean=0.15, std=0.08, gauss_sigma=3, cutout_threshold=0.5, intensity=0.5, mode='rain', p=0.2),
            A.AdvancedBlur(blur_limit=(5, 9), p=0.2),
        ], bbox_params=A.BboxParams(format='yolo', label_fields=['class_ids']))

        if self.level == "light":
            return light
        elif self.level == "medium":
            return medium
        else:
            return hard

    def apply(self, image, bboxes, class_ids):
        transformed = self.transforms(
            image=image,
            bboxes=bboxes,
            class_ids=class_ids
        )
        return transformed["image"], transformed["bboxes"], transformed["class_ids"]

def add_birds_copy_paste(image, objects, max_birds=3):
    if AUG_LEVEL != "hard":
        return image, objects

    h, w = image.shape[:2]
    new_objects = list(objects)

    for _ in range(random.randint(0, max_birds)):
        bird_patch, rel_h, rel_w = get_random_bird_patch()
        if bird_patch is None:
            continue

        scale = random.uniform(0.3, 1.5)
        new_h, new_w = int(bird_patch.shape[0] * scale), int(bird_patch.shape[1] * scale)
        bird_patch = cv2.resize(bird_patch, (new_w, new_h))

        max_x = w - new_w - 10
        max_y = h - new_h - 10
        if max_x <= 0 or max_y <= 0:
            continue

        x, y = random.randint(5, max_x), random.randint(5, max_y)

        alpha = random.uniform(0.6, 0.95)
        roi = image[y:y+new_h, x:x+new_w]
        blended = cv2.addWeighted(roi, 1 - alpha, bird_patch, alpha, 0)
        image[y:y+new_h, x:x+new_w] = blended

        new_xc = (x + new_w / 2) / w
        new_yc = (y + new_h / 2) / h
        new_bw = new_w / w
        new_bh = new_h / h

        new_objects.append({
            "class_id": 1,
            "xc": new_xc, "yc": new_yc, "w": new_bw, "h": new_bh
        })

    return image, new_objects

def augment_dataset():
    print(f"Starting augmentation: level={AUG_LEVEL}, copies_per_image={AUG_COPIES_PER_IMAGE}")
    print(f"Source: {RAW_ROOT}")
    print(f"Output: {AUG_ROOT}")

    if AUG_ROOT.exists():
        shutil.rmtree(AUG_ROOT)

    for split in ["valid", "test"]:
        shutil.copytree(RAW_ROOT / "images" / split, AUG_ROOT / "images" / split)
        shutil.copytree(RAW_ROOT / "labels" / split, AUG_ROOT / "labels" / split)
    print("Valid/Test copied without changes")

    for split in AUGMENT_SPLITS:
        src_imgs = sorted(list((RAW_ROOT / "images" / split).glob("*.jpg")) + 
                         list((RAW_ROOT / "images" / split).glob("*.png")))
        
        aug_img_dir = AUG_ROOT / "images" / f"{split}_aug"
        aug_lbl_dir = AUG_ROOT / "labels" / f"{split}_aug"
        aug_img_dir.mkdir(parents=True)
        aug_lbl_dir.mkdir(parents=True)

        pipeline = AugmentationPipeline(level=AUG_LEVEL)

        stats = {"original": 0, "augmented": 0, "skipped": 0, "bird_paste": 0}

        for img_path in tqdm(src_imgs, desc=f"Augmenting {split}"):
            lbl_path = RAW_ROOT / "labels" / split / f"{img_path.stem}.txt"
            
            dst_img = aug_img_dir / img_path.name
            dst_lbl = aug_lbl_dir / f"{img_path.stem}.txt"
            shutil.copy2(img_path, dst_img)
            if lbl_path.exists():
                shutil.copy2(lbl_path, dst_lbl)
            else:
                dst_lbl.touch()
            stats["original"] += 1

            objects = parse_yolo_label(lbl_path)
            image = cv2.imread(str(img_path))
            if image is None:
                stats["skipped"] += 1
                continue

            bboxes = [[o["xc"], o["yc"], o["w"], o["h"]] for o in objects]
            class_ids = [o["class_id"] for o in objects]

            for aug_idx in range(AUG_COPIES_PER_IMAGE):
                aug_img, aug_bboxes, aug_cls_ids = pipeline.apply(image, bboxes, class_ids)

                if AUG_LEVEL == "hard":
                    aug_img, aug_objects = add_birds_copy_paste(aug_img, objects)
                    if len(aug_objects) > len(objects):
                        stats["bird_paste"] += 1
                    aug_bboxes = [[o["xc"], o["yc"], o["w"], o["h"]] for o in aug_objects]
                    aug_cls_ids = [o["class_id"] for o in aug_objects]

                aug_name = f"{img_path.stem}_aug{aug_idx}{img_path.suffix}"
                cv2.imwrite(str(aug_img_dir / aug_name), aug_img)
                
                aug_objects = [
                    {"class_id": c, "xc": b[0], "yc": b[1], "w": b[2], "h": b[3]}
                    for c, b in zip(aug_cls_ids, aug_bboxes)
                ]
                write_yolo_label(aug_lbl_dir / f"{img_path.stem}_aug{aug_idx}.txt", aug_objects)
                stats["augmented"] += 1

        print(f"\nStatistics {split}:")
        for k, v in stats.items():
            print(f"  {k}: {v}")

    data_yaml = {
        "path": str(AUG_ROOT),
        "train": f"images/{AUGMENT_SPLITS[0]}_aug",
        "val": "images/valid",
        "test": "images/test",
        "names": {
            0: "airplane",
            1: "bird",
            2: "drone",
            3: "helicopter"
        }
    }
    with open(AUG_ROOT / "data.yaml", 'w', encoding='utf-8') as f:
        yaml.dump(data_yaml, f, default_flow_style=False, allow_unicode=True)
    print("data.yaml updated")

if __name__ == "__main__":
    augment_dataset()

🚀 Старт аугментации: уровень=light, копии_на_изображение=1
📂 Источник: C:\Users\IvanC\kurs_work\processed\AOD4_clean
📂 Вывод: C:\Users\IvanC\kurs_work\processed\AOD4_augmented


C:\Users\IvanC\AppData\Local\Temp\ipykernel_7160\1923071964.py:111: UserWarning: Argument(s) 'var_limit' are not valid for transform GaussNoise
  A.GaussNoise(var_limit=(10.0, 30.0), p=0.4),
C:\Users\IvanC\AppData\Local\Temp\ipykernel_7160\1923071964.py:120: UserWarning: Argument(s) 'quality_lower, quality_upper' are not valid for transform ImageCompression
  A.ImageCompression(quality_lower=40, quality_upper=70, p=0.4),
C:\Users\IvanC\AppData\Local\Temp\ipykernel_7160\1923071964.py:122: UserWarning: Argument(s) 'fog_coef_lower, fog_coef_upper' are not valid for transform RandomFog
  A.RandomFog(fog_coef_lower=0.1, fog_coef_upper=0.3, alpha_coef=0.1, p=0.25),
C:\Users\IvanC\AppData\Local\Temp\ipykernel_7160\1923071964.py:124: UserWarning: Argument(s) 'snow_point_lower, snow_point_upper' are not valid for transform RandomSnow
  A.RandomSnow(brightness_coeff=1.1, snow_point_lower=0.1, snow_point_upper=0.25, p=0.2),
C:\Users\IvanC\AppData\Local\Temp\ipykernel_7160\1923071964.py:130: UserW

✅ Valid/Test скопированы без изменений


Augmenting train:   4%|▍         | 634/15761 [00:37<14:57, 16.85it/s]


KeyboardInterrupt: 